## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install torch transformers datasets accelerate scikit-learn tqdm pandas numpy

## 2. Import Libraries

In [ ]:
import json
import os
from pathlib import Path
from typing import List, Dict, Any, Set, Tuple
import random
from collections import Counter, defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset as HFDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm
import pandas as pd
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 3. Configuration

In [ ]:
# Data paths
TAXONOMY_PATH = "../Taxonomy Building/preprocessed_taxonomy.json"
TRAIN_DATA_DIR = "./train_data"
TEST_DATA_DIR = "./test_data"
OUTPUT_DIR = "./hierarchical_model"

# Model selection - Choose one!
MODEL_CHOICE = "deberta"  # Options: "deberta", "scibert"

MODEL_CONFIGS = {
    "deberta": "microsoft/deberta-v3-base",      # ⭐ RECOMMENDED - Best overall
    "scibert": "allenai/scibert_scivocab_uncased" # Good for scientific text
}

MODEL_NAME = MODEL_CONFIGS[MODEL_CHOICE]

# Training hyperparameters
NUM_EPOCHS = 5
BATCH_SIZE = 16  # Encoder models are more efficient
LEARNING_RATE = 2e-5
MAX_LENGTH = 512
WARMUP_RATIO = 0.1

# Hierarchical levels
MAX_HIERARCHY_DEPTH = 4  # Level 1: Domain, Level 2: Field, Level 3: Subfield, Level 4: Topic

print(f"Model: {MODEL_NAME}")
print(f"Max hierarchy depth: {MAX_HIERARCHY_DEPTH}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {NUM_EPOCHS}")

## 4. Load and Process Taxonomy

Extract hierarchical structure and create label mappings for each level

In [ ]:
def extract_hierarchical_paths(taxonomy_dict: Dict, prefix: str = "", level: int = 0) -> List[Tuple[str, List[str]]]:
    """Extract all paths with hierarchical breakdown."""
    paths = []
    
    if isinstance(taxonomy_dict, dict):
        for key, value in taxonomy_dict.items():
            current_path = f"{prefix} > {key}" if prefix else key
            current_hierarchy = prefix.split(' > ') if prefix else []
            current_hierarchy.append(key)
            
            if isinstance(value, dict):
                paths.extend(extract_hierarchical_paths(value, current_path, level + 1))
            elif isinstance(value, list):
                for item in value:
                    leaf_path = f"{current_path} > {item}"
                    leaf_hierarchy = current_hierarchy + [item]
                    paths.append((leaf_path, leaf_hierarchy))
            
            # Also add intermediate paths
            paths.append((current_path, current_hierarchy))
    
    return paths

# Load taxonomy
print("Loading taxonomy...")
with open(TAXONOMY_PATH, 'r', encoding='utf-8') as f:
    taxonomy_data = json.load(f)

taxonomy = taxonomy_data.get('taxonomy', taxonomy_data)

# Extract hierarchical paths
hierarchical_paths = extract_hierarchical_paths(taxonomy)
print(f"Total paths: {len(hierarchical_paths)}")

# Build label mappings for each level
level_labels = defaultdict(set)
for path, hierarchy in hierarchical_paths:
    for level, label in enumerate(hierarchy):
        if level < MAX_HIERARCHY_DEPTH:
            level_labels[level].add(label)

# Create label to ID mappings
label2id = {}
id2label = {}
num_labels_per_level = []

for level in range(MAX_HIERARCHY_DEPTH):
    labels = sorted(list(level_labels[level]))
    label2id[level] = {label: idx for idx, label in enumerate(labels)}
    id2label[level] = {idx: label for idx, label in enumerate(labels)}
    num_labels_per_level.append(len(labels))
    print(f"Level {level + 1}: {len(labels)} unique labels")

print(f"\nLabel counts per level: {num_labels_per_level}")

## 5. Load Training Data

In [ ]:
def load_json_files(data_dir: str) -> List[Dict[str, Any]]:
    """Load all JSON files from directory."""
    all_articles = []
    json_files = list(Path(data_dir).glob("*.json"))
    
    print(f"Found {len(json_files)} JSON files")
    
    for json_file in tqdm(json_files, desc="Loading files"):
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
                if isinstance(data, list):
                    articles = data
                elif isinstance(data, dict):
                    articles = data.get('articles', [data])
                else:
                    continue
                
                valid_articles = [a for a in articles if isinstance(a, dict)]
                all_articles.extend(valid_articles)
                
        except Exception as e:
            print(f"Error loading {json_file.name}: {e}")
            continue
    
    print(f"Total articles loaded: {len(all_articles)}")
    return all_articles

# Load data
print("="*60)
print("LOADING TRAINING DATA")
print("="*60)
train_articles = load_json_files(TRAIN_DATA_DIR)

print("\n" + "="*60)
print("LOADING TEST DATA")
print("="*60)
test_articles = load_json_files(TEST_DATA_DIR)

## 6. Format Data for Hierarchical Classification

In [ ]:
def format_for_hierarchical_classification(articles: List[Dict], label2id: Dict) -> List[Dict]:
    """Format articles for multi-head hierarchical classification."""
    formatted_data = []
    skipped = 0
    
    for article in tqdm(articles, desc="Formatting"):
        # Extract fields
        title = article.get('title', '') or article.get('display_name', '')
        abstract = article.get('abstract', '')
        classification = article.get('classification_path', '')
        
        if not title or not abstract or not classification:
            skipped += 1
            continue
        
        # Clean text
        title = " ".join(title.split()).strip()
        abstract = " ".join(abstract.split()).strip()
        
        # Parse hierarchy
        hierarchy = [part.strip() for part in classification.split(' > ')]
        
        # Convert to label IDs for each level
        label_ids = []
        valid = True
        
        for level in range(MAX_HIERARCHY_DEPTH):
            if level < len(hierarchy):
                label = hierarchy[level]
                if label in label2id[level]:
                    label_ids.append(label2id[level][label])
                else:
                    valid = False
                    break
            else:
                # Pad with -100 (ignore index)
                label_ids.append(-100)
        
        if not valid:
            skipped += 1
            continue
        
        formatted_data.append({
            'text': f"{title} [SEP] {abstract}",
            'labels': label_ids,
            'classification': classification,
            'hierarchy': hierarchy
        })
    
    print(f"Formatted {len(formatted_data)} articles")
    print(f"Skipped {skipped} articles")
    return formatted_data

# Format data
train_formatted = format_for_hierarchical_classification(train_articles, label2id)
test_formatted = format_for_hierarchical_classification(test_articles, label2id)

# Split training data
train_data, val_data = train_test_split(train_formatted, test_size=0.1, random_state=42)

print(f"\nFinal dataset sizes:")
print(f"  Training:   {len(train_data):,}")
print(f"  Validation: {len(val_data):,}")
print(f"  Test:       {len(test_formatted):,}")

## 7. Define Multi-Head Hierarchical Model

In [ ]:
class HierarchicalClassifier(nn.Module):
    """Multi-head hierarchical classification model."""
    
    def __init__(self, model_name: str, num_labels_per_level: List[int], dropout: float = 0.1):
        super().__init__()
        
        # Load base model
        self.encoder = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.encoder.config.hidden_size
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Classification heads for each level
        self.classifiers = nn.ModuleList([
            nn.Linear(self.hidden_size, num_labels)
            for num_labels in num_labels_per_level
        ])
        
        self.num_levels = len(num_labels_per_level)
    
    def forward(self, input_ids, attention_mask, labels=None):
        # Encode
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # Get [CLS] token representation
        pooled_output = outputs.last_hidden_state[:, 0, :]  # [batch_size, hidden_size]
        pooled_output = self.dropout(pooled_output)
        
        # Multi-head classification
        logits = [classifier(pooled_output) for classifier in self.classifiers]
        
        # Calculate loss if labels provided
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            losses = []
            
            for level in range(self.num_levels):
                level_labels = labels[:, level]
                level_logits = logits[level]
                level_loss = loss_fct(level_logits, level_labels)
                losses.append(level_loss)
            
            # Weighted average (give more weight to deeper levels)
            weights = torch.tensor([1.0, 2.0, 3.0, 4.0][:self.num_levels], device=labels.device)
            weights = weights / weights.sum()
            loss = sum(w * l for w, l in zip(weights, losses))
        
        return {'loss': loss, 'logits': logits}

print("✅ Hierarchical classifier defined!")

## 8. Create Dataset Class

In [ ]:
class HierarchicalDataset(Dataset):
    def __init__(self, data: List[Dict], tokenizer, max_length: int):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Tokenize
        encoding = self.tokenizer(
            item['text'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(item['labels'], dtype=torch.long)
        }

print("✅ Dataset class defined!")

## 9. Initialize Model and Tokenizer

In [ ]:
# Load tokenizer
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Initialize model
print(f"Loading model: {MODEL_NAME}")
model = HierarchicalClassifier(
    model_name=MODEL_NAME,
    num_labels_per_level=num_labels_per_level
)

# Move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Model initialized!")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Device: {device}")

## 10. Create Datasets and DataLoaders

In [ ]:
# Create datasets
train_dataset = HierarchicalDataset(train_data, tokenizer, MAX_LENGTH)
val_dataset = HierarchicalDataset(val_data, tokenizer, MAX_LENGTH)
test_dataset = HierarchicalDataset(test_formatted, tokenizer, MAX_LENGTH)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"✅ Datasets created!")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

## 11. Training Setup

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import get_linear_schedule_with_warmup

# Optimizer
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# Scheduler
num_training_steps = len(train_loader) * NUM_EPOCHS
num_warmup_steps = int(num_training_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

print(f"✅ Training setup complete!")
print(f"  Total training steps: {num_training_steps}")
print(f"  Warmup steps: {num_warmup_steps}")
print(f"  Learning rate: {LEARNING_RATE}")

## 12. Training Loop

In [ ]:
def evaluate(model, dataloader, device):
    """Evaluate model on validation/test set."""
    model.eval()
    total_loss = 0
    all_predictions = [[] for _ in range(MAX_HIERARCHY_DEPTH)]
    all_labels = [[] for _ in range(MAX_HIERARCHY_DEPTH)]
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask, labels)
            total_loss += outputs['loss'].item()
            
            # Get predictions for each level
            for level in range(MAX_HIERARCHY_DEPTH):
                logits = outputs['logits'][level]
                preds = torch.argmax(logits, dim=-1)
                
                all_predictions[level].extend(preds.cpu().numpy())
                all_labels[level].extend(labels[:, level].cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    
    # Calculate accuracy for each level
    accuracies = []
    for level in range(MAX_HIERARCHY_DEPTH):
        # Filter out -100 labels
        valid_indices = [i for i, label in enumerate(all_labels[level]) if label != -100]
        if valid_indices:
            level_preds = [all_predictions[level][i] for i in valid_indices]
            level_labels = [all_labels[level][i] for i in valid_indices]
            accuracy = accuracy_score(level_labels, level_preds)
            accuracies.append(accuracy)
        else:
            accuracies.append(0.0)
    
    return avg_loss, accuracies, all_predictions, all_labels

# Training loop
print("="*60)
print("STARTING TRAINING")
print("="*60)

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 60)
    
    # Training
    model.train()
    total_train_loss = 0
    
    for batch in tqdm(train_loader, desc="Training"):
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask, labels)
        loss = outputs['loss']
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_train_loss += loss.item()
    
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation
    val_loss, val_accuracies, _, _ = evaluate(model, val_loader, device)
    
    # Save history
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_accuracies)
    
    # Print results
    print(f"\nTrain Loss: {avg_train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val Accuracies:")
    for level, acc in enumerate(val_accuracies):
        print(f"  Level {level + 1}: {acc:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best_model.pt")
        print(f"✅ Saved best model (val_loss: {val_loss:.4f})")

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)

## 13. Evaluate on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best_model.pt"))
print("Loaded best model for evaluation")

# Evaluate
test_loss, test_accuracies, test_predictions, test_labels = evaluate(model, test_loader, device)

print("\n" + "="*60)
print("TEST SET RESULTS")
print("="*60)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"\nHierarchical Accuracies:")
for level, acc in enumerate(test_accuracies):
    level_name = ["Domain", "Field", "Subfield", "Topic"][level] if level < 4 else f"Level {level+1}"
    print(f"  {level_name:12s}: {acc:.4f} ({acc*100:.2f}%)")

## 14. Detailed Per-Level Analysis

In [ ]:
# Per-level classification reports
for level in range(MAX_HIERARCHY_DEPTH):
    print(f"\n{'='*60}")
    print(f"LEVEL {level + 1} CLASSIFICATION REPORT")
    print(f"{'='*60}")
    
    # Filter valid predictions
    valid_indices = [i for i, label in enumerate(test_labels[level]) if label != -100]
    
    if not valid_indices:
        print("No valid samples for this level")
        continue
    
    level_preds = [test_predictions[level][i] for i in valid_indices]
    level_labels = [test_labels[level][i] for i in valid_indices]
    
    # Get label names
    unique_labels = sorted(set(level_labels))
    target_names = [id2label[level][label_id] for label_id in unique_labels]
    
    # Classification report
    report = classification_report(
        level_labels,
        level_preds,
        labels=unique_labels,
        target_names=target_names,
        zero_division=0
    )
    
    print(report)

## 15. Save Model and Results

In [ ]:
# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save model
torch.save(model.state_dict(), f"{OUTPUT_DIR}/final_model.pt")
tokenizer.save_pretrained(OUTPUT_DIR)

# Save label mappings
label_mappings = {
    'label2id': {f'level_{i}': mapping for i, mapping in label2id.items()},
    'id2label': {f'level_{i}': mapping for i, mapping in id2label.items()},
    'num_labels_per_level': num_labels_per_level
}

with open(f"{OUTPUT_DIR}/label_mappings.json", 'w', encoding='utf-8') as f:
    json.dump(label_mappings, f, indent=2, ensure_ascii=False)

# Save training history
with open(f"{OUTPUT_DIR}/training_history.json", 'w') as f:
    json.dump(history, f, indent=2)

# Save evaluation results
results = {
    'model': MODEL_NAME,
    'test_loss': float(test_loss),
    'test_accuracies': {
        f'level_{i+1}': float(acc) 
        for i, acc in enumerate(test_accuracies)
    },
    'hyperparameters': {
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'num_epochs': NUM_EPOCHS,
        'max_length': MAX_LENGTH
    }
}

with open(f"{OUTPUT_DIR}/evaluation_results.json", 'w') as f:
    json.dump(results, f, indent=2)

print("✅ Model and results saved!")
print(f"\nFiles saved to: {OUTPUT_DIR}/")
print("  • final_model.pt")
print("  • best_model.pt")
print("  • label_mappings.json")
print("  • training_history.json")
print("  • evaluation_results.json")
print("  • tokenizer files")

## 16. Inference Function for Production

In [ ]:
def predict_classification(model, tokenizer, title: str, abstract: str, device, label2id, id2label):
    """Predict hierarchical classification for a single article."""
    model.eval()
    
    # Prepare input
    text = f"{title} [SEP] {abstract}"
    encoding = tokenizer(
        text,
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
        logits = outputs['logits']
    
    # Get predictions and confidence scores
    predictions = []
    confidence_scores = []
    
    for level in range(len(logits)):
        probs = torch.softmax(logits[level], dim=-1)[0]
        pred_id = torch.argmax(probs).item()
        confidence = probs[pred_id].item()
        
        pred_label = id2label[level][pred_id]
        predictions.append(pred_label)
        confidence_scores.append(confidence)
    
    # Build full path
    full_path = " > ".join(predictions)
    avg_confidence = sum(confidence_scores) / len(confidence_scores)
    
    return {
        'classification': full_path,
        'hierarchy': predictions,
        'confidence_scores': confidence_scores,
        'average_confidence': avg_confidence
    }

# Test inference
sample = test_formatted[0]
title, abstract = sample['text'].split(' [SEP] ')

result = predict_classification(model, tokenizer, title, abstract, device, label2id, id2label)

print("\n" + "="*60)
print("SAMPLE PREDICTION")
print("="*60)
print(f"\nTitle: {title[:100]}...")
print(f"\nPredicted Classification: {result['classification']}")
print(f"\nTrue Classification: {sample['classification']}")
print(f"\nConfidence Scores:")
for level, (label, conf) in enumerate(zip(result['hierarchy'], result['confidence_scores'])):
    print(f"  Level {level+1} ({label}): {conf:.4f}")
print(f"\nAverage Confidence: {result['average_confidence']:.4f}")

## 17. Example: Batch Prediction

In [ ]:
# Predict on 10 random test samples
print("="*60)
print("BATCH PREDICTION EXAMPLES")
print("="*60)

random_samples = random.sample(test_formatted, min(10, len(test_formatted)))

for i, sample in enumerate(random_samples):
    title, abstract = sample['text'].split(' [SEP] ')
    result = predict_classification(model, tokenizer, title, abstract, device, label2id, id2label)
    
    print(f"\n{'='*60}")
    print(f"Example {i+1}")
    print(f"{'='*60}")
    print(f"Title: {title[:80]}...")
    print(f"\nPredicted: {result['classification']}")
    print(f"True:      {sample['classification']}")
    print(f"Confidence: {result['average_confidence']:.2%}")
    
    if result['classification'] == sample['classification']:
        print("✅ CORRECT")
    else:
        print("❌ INCORRECT")

## Summary

### ✅ What We Built:
- Multi-head hierarchical classification model
- Separate classification head for each taxonomy level
- Weighted loss function (more weight on deeper levels)
- Confidence scores for each level

### 🎯 Benefits vs Causal LLMs:
- **140M parameters** vs 7B (50x smaller!)
- **10-20x faster** inference
- **Better accuracy** for classification tasks
- **Lower memory** requirements
- **Hierarchical structure** enforced by architecture

### 📊 Next Steps:
1. **Integrate with RAG**: Use this model with retrieved taxonomy paths
2. **Deploy to production**: Create FastAPI endpoint
3. **Monitor performance**: Track accuracy per domain
4. **Fine-tune further**: Use domain-specific data

### 🚀 For Production Use:
```python
# Load model
model = HierarchicalClassifier(MODEL_NAME, num_labels_per_level)
model.load_state_dict(torch.load('hierarchical_model/final_model.pt'))
tokenizer = AutoTokenizer.from_pretrained('hierarchical_model/')

# Predict
result = predict_classification(model, tokenizer, title, abstract, device, label2id, id2label)
print(result['classification'])
```